# D7 Stage 2c Acceptance Notebook

This notebook is the independent acceptance evidence for the D7 Stage 2c twenty-call live D7b probe. It is not a presentation layer for the fire script. It mechanically audits implementation artifacts first, then adjudicates whether Stage 2d receives a green light.

Layer A covers implementation and artifact integrity. Layer B covers scientific interpretation and go/no-go criteria.


## BR. Stage 2c Scope and Artifact Framing

Fix the artifact set, load the live-fire record, and verify the anti-hindsight timestamp chain before any analysis is trusted.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sqlite3
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean, median

import pandas as pd
from IPython.display import Markdown, display

START_CWD = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (START_CWD, *START_CWD.parents) if (p / 'pyproject.toml').exists() and (p / 'agents').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repo root from {START_CWD}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

BATCH_UUID = '5cf76668-47d1-48d7-bd90-db06d31982ed'
N = 20
OVERLAP_POSITIONS = {17, 73, 74, 97, 138}
ALLOWED_AMBIGUITY_SOURCES = {'test-retest variance', 'theme imbalance', 'overlap confounding', 'label sparsity'}

SELECTION_PATH = Path('docs/d7_stage2c/replay_candidates.json')
EXPECTATIONS_PATH = Path('docs/d7_stage2c/stage2c_expectations.md')
AGGREGATE_PATH = Path('docs/d7_stage2c/stage2c_batch_record.json')
PER_CALL_DIR = Path('docs/d7_stage2c')
PER_CALL_PATHS = [PER_CALL_DIR / f'call_{i}_live_call_record.json' for i in range(1, N + 1)]
LEDGER_PATH = Path('agents/spend_ledger.db')
PROMPT_TEMPLATE_PATH = Path('agents/critic/d7b_prompt.py')
RAW_CRITIC_DIR = Path('raw_payloads') / f'batch_{BATCH_UUID}' / 'critic'
STAGE2B_AGGREGATE_PATH = Path('docs/d7_stage2b/stage2b_batch_record.json')

def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

def git_commit_ts(path: Path) -> int:
    out = subprocess.check_output(['git', 'log', '-1', '--format=%ct', '--', str(path)], text=True).strip()
    if not out:
        raise AssertionError(f'No git commit timestamp for {path}')
    return int(out)

def git_commit_iso(path: Path) -> str:
    return datetime.fromtimestamp(git_commit_ts(path), tz=timezone.utc).isoformat().replace('+00:00', 'Z')

def parse_iso(value: str) -> datetime:
    if value.endswith('Z'):
        value = value[:-1] + '+00:00'
    return datetime.fromisoformat(value)

def approx_equal(a: float, b: float, tol: float = 1e-9) -> bool:
    return abs(float(a) - float(b)) <= tol

def status_pass(scan: dict | None) -> bool:
    return isinstance(scan, dict) and scan.get('status') == 'pass'

def d7b_scores(rec: dict) -> dict:
    return rec.get('critic_result', {}).get('d7b_llm_scores') or {}

def expected_candidate_header(i: int, c: dict) -> str:
    return f"### Candidate {i} \u2014 Position {c['position']}, {c['theme']}, {c['d7a_b_relationship_label']}"

def extract_neutral_group_body(text: str) -> str | None:
    header = '## Aggregate Prediction for Neutral Group'
    lines = text.splitlines(keepends=True)
    start = None
    for i, line in enumerate(lines):
        if line.rstrip('\r\n') == header:
            start = i + 1
            break
    if start is None:
        return None
    end = len(lines)
    for j in range(start, len(lines)):
        if re.match(r'^##\s', lines[j]):
            end = j
            break
    return ''.join(lines[start:end])

def round6(values):
    return [round(float(v or 0), 6) for v in values]

core_artifacts = [SELECTION_PATH, EXPECTATIONS_PATH, AGGREGATE_PATH, LEDGER_PATH, PROMPT_TEMPLATE_PATH]
missing = [str(p) for p in [*core_artifacts, *PER_CALL_PATHS] if not p.exists()]
assert not missing, 'Missing Stage 2c acceptance artifacts: ' + ', '.join(missing)

selection_raw = load_json(SELECTION_PATH)
candidates = selection_raw if isinstance(selection_raw, list) else selection_raw['candidates']
expectations_text = EXPECTATIONS_PATH.read_text(encoding='utf-8')
aggregate = load_json(AGGREGATE_PATH)
per_calls = [load_json(p) for p in PER_CALL_PATHS]

selection_ts = git_commit_ts(SELECTION_PATH)
expectations_ts = git_commit_ts(EXPECTATIONS_PATH)
fire_start_ts = int(parse_iso(aggregate['fire_timestamp_utc_start']).timestamp())

assert aggregate['stage_label'] == 'd7_stage2c'
assert aggregate['record_version'] == '1.0'
assert aggregate['batch_uuid'] == BATCH_UUID
for key in ('selection_json_sha256', 'expectations_file_sha256', 'd7b_prompt_template_sha256'):
    assert aggregate.get(key), f'Missing aggregate SHA field: {key}'
assert selection_ts < expectations_ts < fire_start_ts

artifact_df = pd.DataFrame([
    {'artifact': str(SELECTION_PATH), 'exists': SELECTION_PATH.exists(), 'sha256': sha256_file(SELECTION_PATH)},
    {'artifact': str(EXPECTATIONS_PATH), 'exists': EXPECTATIONS_PATH.exists(), 'sha256': sha256_file(EXPECTATIONS_PATH)},
    {'artifact': str(AGGREGATE_PATH), 'exists': AGGREGATE_PATH.exists(), 'sha256': sha256_file(AGGREGATE_PATH)},
    *[{'artifact': str(p), 'exists': p.exists(), 'sha256': sha256_file(p)} for p in PER_CALL_PATHS],
])
display(artifact_df)
print('selection commit:', git_commit_iso(SELECTION_PATH))
print('expectations commit:', git_commit_iso(EXPECTATIONS_PATH))
print('fire start:', aggregate['fire_timestamp_utc_start'])


,artifact,exists,sha256
0,docs/d7_stage2c/replay_candidates.json,True,17254003cf93a958cebc0ad26671da59aa166ff8de8063...
1,docs/d7_stage2c/stage2c_expectations.md,True,14aaefafa958eee29f771d3c0b49db317ce8dac0d7802f...
2,docs/d7_stage2c/stage2c_batch_record.json,True,c4a4072f61326dc23ba5d09c6263b5db5a0c08d2737dba...
3,docs/d7_stage2c/call_1_live_call_record.json,True,5cd724626d9357c16dfed46466ee0be1d9c0416efb9595...
4,docs/d7_stage2c/call_2_live_call_record.json,True,2ef14194782860dfa728015c51e2af3e6c21da83c8b27e...
5,docs/d7_stage2c/call_3_live_call_record.json,True,fe0f06c6c77d9db09c67ed65106a9ae1e4e7ebdc498753...
6,docs/d7_stage2c/call_4_live_call_record.json,True,ff898ed4e75ef2707a09dd51b5c2aa63e11515406a33b9...
7,docs/d7_stage2c/call_5_live_call_record.json,True,ab2929bcde2875e0a7b2add2adb1fd92c905bd90c8b2eb...
8,docs/d7_stage2c/call_6_live_call_record.json,True,c549293be61cc97e742777de593fd53140191e7d8d2084...
9,docs/d7_stage2c/call_7_live_call_record.json,True,0bf7719d396f1e5d70d64bf81074af09e28c0c65af0e38...


selection commit: 2026-04-18T15:20:46Z
expectations commit: 2026-04-19T14:05:22Z
fire start: 2026-04-19T14:25:43.024994Z


## BS. Selection Lock Audit

Rebuild the Stage 2c selection contract directly from `replay_candidates.json`: count, order, diversity, labels, buckets, and Stage 2b overlap identity.


In [2]:
assert len(candidates) == N
firing_orders = [c['firing_order'] for c in candidates]
positions = [c['position'] for c in candidates]
hashes = [c['hypothesis_hash'] for c in candidates]
themes = [c['theme'] for c in candidates]
labels = [c['d7a_b_relationship_label'] for c in candidates]
buckets = [c['position_bucket'] for c in candidates]

assert firing_orders == list(range(1, N + 1))
assert positions == sorted(positions)
assert len(set(hashes)) == N
assert len(set(themes)) >= 3
assert all(v >= 5 for v in Counter(buckets).values())
assert Counter(labels)['agreement_expected'] >= 4
assert {c['position'] for c in candidates if c['position'] in OVERLAP_POSITIONS} == OVERLAP_POSITIONS
assert [p for p in positions if p in OVERLAP_POSITIONS] == [17, 73, 74, 97, 138]

selection_label_counts = dict(Counter(labels))
selection_theme_counts = dict(Counter(themes))
selection_bucket_counts = dict(Counter(buckets))
selection_divergence_count = sum(1 for c in candidates if c['d7a_b_relationship_label'] == 'divergence_expected')
assert selection_divergence_count == selection_label_counts.get('divergence_expected', 0)

selection_df = pd.DataFrame([
    {
        'firing_order': c['firing_order'],
        'position': c['position'],
        'theme': c['theme'],
        'bucket': c['position_bucket'],
        'label': c['d7a_b_relationship_label'],
        'hash': c['hypothesis_hash'],
        'stage2b_overlap': c['position'] in OVERLAP_POSITIONS,
    }
    for c in candidates
])
display(selection_df)
display(pd.DataFrame([
    {'metric': 'theme_counts_in_selected', 'value': selection_theme_counts},
    {'metric': 'label_counts_in_selected', 'value': selection_label_counts},
    {'metric': 'bucket_counts_in_selected', 'value': selection_bucket_counts},
    {'metric': 'stage2b_overlap_positions', 'value': sorted(OVERLAP_POSITIONS)},
]))


,firing_order,position,theme,bucket,label,hash,stage2b_overlap
0,1,17,mean_reversion,early,divergence_expected,17396a8e291dae75,True
1,2,22,mean_reversion,early,neutral,dd38cf9547fa9306,False
2,3,27,mean_reversion,early,agreement_expected,57f884d5e47b14e6,False
3,4,32,mean_reversion,early,neutral,663b1d38277a85f9,False
4,5,62,mean_reversion,early,neutral,ed50f3d2012b510d,False
5,6,72,mean_reversion,mid,neutral,ca08df07cc55d138,False
6,7,73,volatility_regime,mid,divergence_expected,ab3c1725e1cf4890,True
7,8,74,volume_divergence,mid,divergence_expected,eb019d8da279abb2,True
8,9,77,mean_reversion,mid,neutral,aa594fe2f732f9ba,False
9,10,83,volatility_regime,mid,neutral,00fe887f948bb83b,False


,metric,value
0,theme_counts_in_selected,"{'mean_reversion': 15, 'volatility_regime': 4,..."
1,label_counts_in_selected,"{'divergence_expected': 3, 'neutral': 9, 'agre..."
2,bucket_counts_in_selected,"{'early': 5, 'mid': 10, 'late': 5}"
3,stage2b_overlap_positions,"[17, 73, 74, 97, 138]"


## BT. Expectations File Structural Audit

Audit anti-hindsight structure only. This checks exact headers and the neutral-group aggregate prediction body length, not the quality of Charlie's prediction.


In [3]:
required_headers = [
    '## Anti-Hindsight Anchor',
    '## Aggregate Expectations Across All 20 Calls',
    '## Per-Candidate Expectations',
    '## Aggregate Prediction for Neutral Group',
]
for header in required_headers:
    assert header in expectations_text, f'Missing expectations header: {header}'

candidate_headers = [expected_candidate_header(i, c) for i, c in enumerate(candidates, 1)]
missing_headers = [h for h in candidate_headers if h not in expectations_text]
assert not missing_headers, 'Missing exact candidate headers: ' + repr(missing_headers)

neutral_group_body = extract_neutral_group_body(expectations_text)
assert neutral_group_body is not None
neutral_non_ws = re.sub(r'\s+', '', neutral_group_body)
assert len(neutral_non_ws) >= 50
assert selection_ts < expectations_ts < fire_start_ts

expectations_audit_df = pd.DataFrame([
    {'check': 'required headers present', 'pass': all(h in expectations_text for h in required_headers)},
    {'check': '20 exact candidate headers present', 'pass': len(missing_headers) == 0},
    {'check': 'neutral-group body non-whitespace chars >= 50', 'pass': len(neutral_non_ws) >= 50, 'value': len(neutral_non_ws)},
    {'check': 'selection < expectations < fire_start', 'pass': selection_ts < expectations_ts < fire_start_ts},
])
display(expectations_audit_df)


,check,pass,value
0,required headers present,True,NaN
1,20 exact candidate headers present,True,NaN
2,neutral-group body non-whitespace chars >= 50,True,1134.0
3,selection < expectations < fire_start,True,NaN


## BU. Aggregate Record Schema Audit

Audit aggregate schema and independently rebuild totals and ordered sequences from the 20 per-call record files, not from embedded aggregate summaries.


In [4]:
assert aggregate['completed_call_count'] == N
assert aggregate['sequence_aborted'] is False
assert aggregate['abort_reason'] is None
assert aggregate['abort_at_call_index'] is None
assert len(aggregate['reasoning_lengths_in_call_order']) == N
assert len(aggregate['actual_costs_in_call_order']) == N
assert len(aggregate['critic_statuses_in_call_order']) == N
assert len(aggregate['per_call_records']) == N
assert 'write_completed_at' in aggregate
assert aggregate['stage2b_overlap_count'] == 5
assert aggregate['stage2b_overlap_completed_count'] <= 5
assert aggregate['stage2b_overlap_positions'] == sorted(OVERLAP_POSITIONS)

recomputed_reasoning_lengths = [len(rec.get('critic_result', {}).get('d7b_reasoning') or '') for rec in per_calls]
recomputed_actual_costs = round6([rec['actual_cost_usd'] for rec in per_calls])
recomputed_estimated_costs = round6([rec['cost']['estimated_usd'] for rec in per_calls])
recomputed_input_tokens = [rec['critic_result'].get('d7b_input_tokens') for rec in per_calls]
recomputed_output_tokens = [rec['critic_result'].get('d7b_output_tokens') for rec in per_calls]
recomputed_statuses = [rec['critic_status'] for rec in per_calls]
recomputed_labels = [rec['pre_registered_label'] for rec in per_calls]
recomputed_themes = [rec['candidate_theme'] for rec in per_calls]
recomputed_overlap_seq = [rec['candidate_position'] in OVERLAP_POSITIONS for rec in per_calls]

assert recomputed_reasoning_lengths == aggregate['reasoning_lengths_in_call_order']
assert recomputed_actual_costs == aggregate['actual_costs_in_call_order']
assert recomputed_estimated_costs == aggregate['estimated_costs_in_call_order']
assert recomputed_input_tokens == aggregate['input_tokens_in_call_order']
assert recomputed_output_tokens == aggregate['output_tokens_in_call_order']
assert recomputed_statuses == aggregate['critic_statuses_in_call_order']
assert dict(Counter(recomputed_labels)) == aggregate['label_counts_in_sequence']
assert dict(Counter(recomputed_themes)) == aggregate['theme_counts_in_sequence']
assert recomputed_overlap_seq == aggregate['is_stage2b_overlap_in_call_order']
assert round(sum(recomputed_actual_costs), 6) == aggregate['total_actual_cost_usd']
assert round(sum(recomputed_estimated_costs), 6) == aggregate['total_estimated_cost_usd']
assert sum(recomputed_input_tokens) == aggregate['total_input_tokens']
assert sum(recomputed_output_tokens) == aggregate['total_output_tokens']

for label, expected_count in selection_label_counts.items():
    bucket = aggregate['svr_by_label'][label]
    assert bucket['completed_count'] == expected_count
    assert len(bucket['positions']) == len(bucket['svr_values']) == expected_count
    assert bucket['positions'] == sorted(bucket['positions'])

aggregate_rebuild_df = pd.DataFrame({
    'call': list(range(1, N + 1)),
    'reasoning_len_rebuilt': recomputed_reasoning_lengths,
    'reasoning_len_aggregate': aggregate['reasoning_lengths_in_call_order'],
    'actual_cost_rebuilt': recomputed_actual_costs,
    'actual_cost_aggregate': aggregate['actual_costs_in_call_order'],
    'input_tokens_rebuilt': recomputed_input_tokens,
    'input_tokens_aggregate': aggregate['input_tokens_in_call_order'],
    'output_tokens_rebuilt': recomputed_output_tokens,
    'output_tokens_aggregate': aggregate['output_tokens_in_call_order'],
})
display(aggregate_rebuild_df)


,call,reasoning_len_rebuilt,reasoning_len_aggregate,actual_cost_rebuilt,actual_cost_aggregate,input_tokens_rebuilt,input_tokens_aggregate,output_tokens_rebuilt,output_tokens_aggregate
0,1,1061,1061,0.011037,0.011037,1909,1909,354,354
1,2,1192,1192,0.011943,0.011943,2041,2041,388,388
2,3,1033,1033,0.011280,0.011280,2180,2180,316,316
3,4,1120,1120,0.011859,0.011859,2278,2278,335,335
4,5,1134,1134,0.014727,0.014727,2969,2969,388,388
5,6,1029,1029,0.014223,0.014223,3206,3206,307,307
6,7,1032,1032,0.014331,0.014331,3192,3192,317,317
7,8,1018,1018,0.014937,0.014937,3204,3204,355,355
8,9,1106,1106,0.014784,0.014784,3273,3273,331,331
9,10,1059,1059,0.014922,0.014922,3444,3444,306,306


## BV. Per-Call Record Audit

Verify every per-call record against the locked selection row, raw response presence, scan status, ledger row mirror, cost ceiling, and reasoning-length band.


In [5]:
selection_by_order = {c['firing_order']: c for c in candidates}
per_call_rows = []
prior_counts = []

for path, rec in zip(PER_CALL_PATHS, per_calls):
    assert path.exists(), f'Missing per-call record: {path}'
    order = rec['firing_order']
    c = selection_by_order[order]
    scores = d7b_scores(rec)
    reasoning_len = len(rec.get('critic_result', {}).get('d7b_reasoning') or '')
    prior_counts.append(rec['prior_factor_sets_count'])

    assert rec['critic_status'] in {'ok', 'd7b_error'}
    assert rec['candidate_position'] == c['position']
    assert rec['candidate_theme'] == c['theme']
    assert rec['pre_registered_label'] == c['d7a_b_relationship_label']
    assert rec['is_stage2b_overlap'] == (c['position'] in OVERLAP_POSITIONS)
    assert re.fullmatch(r'[0-9a-f]{64}', rec['prompt_sha256'])
    assert rec['prior_factor_sets_count'] >= 0
    assert rec['theme_hint_factor_count'] >= 0
    assert rec['actual_cost_usd'] <= 0.05
    assert 100 <= reasoning_len <= 1500
    assert status_pass(rec['forbidden_language_scan_result'])
    assert status_pass(rec['refusal_scan_result'])
    assert status_pass(rec['leakage_audit_result'])
    assert rec['ledger_row']['batch_id'] == BATCH_UUID
    assert rec['ledger_row']['api_call_kind'] == 'd7b_critic_live'

    if rec['critic_status'] == 'ok':
        response_path = Path(rec['raw_payload_paths']['response'])
        assert response_path.exists(), f'Missing ok-path raw response: {response_path}'

    per_call_rows.append({
        'call': order,
        'position': rec['candidate_position'],
        'theme': rec['candidate_theme'],
        'label': rec['pre_registered_label'],
        'status': rec['critic_status'],
        'is_stage2b_overlap': rec['is_stage2b_overlap'],
        'actual_cost': rec['actual_cost_usd'],
        'reasoning_len': reasoning_len,
        'svr': scores.get('structural_variant_risk'),
        'ledger_row_id': rec['ledger_row']['row_id'],
    })

assert sorted(rec['firing_order'] for rec in per_calls) == list(range(1, N + 1))
assert prior_counts[-1] >= prior_counts[0]
per_call_df = pd.DataFrame(per_call_rows)
display(per_call_df)


,call,position,theme,label,status,is_stage2b_overlap,actual_cost,reasoning_len,svr,ledger_row_id
0,1,17,mean_reversion,divergence_expected,ok,True,0.011037,1061,0.85,fb3eeb31-6e1a-4f07-b4be-73b14cfb1556
1,2,22,mean_reversion,neutral,ok,False,0.011943,1192,0.85,dcc89d70-535f-45f1-b564-2ea2f94b935f
2,3,27,mean_reversion,agreement_expected,ok,False,0.011280,1033,0.85,37d90694-735c-44c5-ba23-b1c785c50b5f
3,4,32,mean_reversion,neutral,ok,False,0.011859,1120,0.90,075f4bfa-0a1e-46fd-b072-ff03896311c9
4,5,62,mean_reversion,neutral,ok,False,0.014727,1134,0.95,b2936740-443e-4c30-9db8-7550d8c4782a
5,6,72,mean_reversion,neutral,ok,False,0.014223,1029,0.75,96efc334-e73f-4ed4-9e85-2f96d85b31ca
6,7,73,volatility_regime,divergence_expected,ok,True,0.014331,1032,0.95,ce56b686-ff9d-4b80-92fd-a4dd4907d5ea
7,8,74,volume_divergence,divergence_expected,ok,True,0.014937,1018,0.75,22c69d43-fd71-468f-a169-bbb59063018a
8,9,77,mean_reversion,neutral,ok,False,0.014784,1106,0.95,06da54b6-d7c0-4012-a5b6-d72b52902064
9,10,83,volatility_regime,neutral,ok,False,0.014922,1059,0.75,d1d6803b-e2e1-42e6-9e72-accc24db705f


## BW. Raw Payload and Stage 2b Archival Audit

Verify Stage 2b overlap archival separately from new Stage 2c payload presence. The five overlap positions must have old artifacts in `stage2b_archive/` and new artifacts in the live critic directory.


In [6]:
assert RAW_CRITIC_DIR.exists(), f'Missing raw critic directory: {RAW_CRITIC_DIR}'
archive_dir = RAW_CRITIC_DIR / 'stage2b_archive'
assert archive_dir.exists(), f'Missing Stage 2b archive directory: {archive_dir}'

raw_rows = []
for pos in positions:
    main_prompt = RAW_CRITIC_DIR / f'call_{pos:04d}_prompt.txt'
    main_response = RAW_CRITIC_DIR / f'call_{pos:04d}_response.json'
    main_result = RAW_CRITIC_DIR / f'call_{pos:04d}_critic_result.json'
    assert main_prompt.exists(), f'Missing Stage 2c prompt payload for {pos}'
    assert main_response.exists(), f'Missing Stage 2c response payload for {pos}'
    assert main_result.exists(), f'Missing Stage 2c critic_result payload for {pos}'

    archived_files = sorted(archive_dir.glob(f'call_{pos:04d}_*'))
    if pos in OVERLAP_POSITIONS:
        archived_names = {p.name for p in archived_files}
        assert any(name.endswith('_prompt.txt') for name in archived_names), f'Missing archived prompt for overlap {pos}'
        assert any(name.endswith('_response.json') for name in archived_names), f'Missing archived response for overlap {pos}'
        assert any(name.endswith('_critic_result.json') for name in archived_names), f'Missing archived critic_result for overlap {pos}'
    else:
        assert not archived_files, f'Unexpected archived non-overlap artifacts for {pos}'

    raw_rows.append({
        'position': pos,
        'stage2c_main_prompt': main_prompt.exists(),
        'stage2c_main_response': main_response.exists(),
        'stage2c_main_critic_result': main_result.exists(),
        'stage2b_archive_file_count': len(archived_files),
        'overlap': pos in OVERLAP_POSITIONS,
    })

raw_payload_df = pd.DataFrame(raw_rows)
display(raw_payload_df)


,position,stage2c_main_prompt,stage2c_main_response,stage2c_main_critic_result,stage2b_archive_file_count,overlap
0,17,True,True,True,3,True
1,22,True,True,True,0,False
2,27,True,True,True,0,False
3,32,True,True,True,0,False
4,62,True,True,True,0,False
5,72,True,True,True,0,False
6,73,True,True,True,3,True
7,74,True,True,True,3,True
8,77,True,True,True,0,False
9,83,True,True,True,0,False


## BX. Ledger Audit

Query SQLite directly and reconcile the twenty live Stage 2c rows against per-call records and aggregate totals.


In [7]:
conn = sqlite3.connect(LEDGER_PATH)
ledger_rows = conn.execute(
    """
    SELECT id, batch_id, api_call_kind, backend_kind, call_role, status,
           estimated_cost, actual_cost, input_tokens, output_tokens,
           created_at_utc, completed_at_utc, notes
    FROM ledger
    WHERE batch_id = ?
      AND backend_kind = 'd7b_critic'
      AND api_call_kind = 'd7b_critic_live'
      AND notes LIKE 'Stage 2c%'
      AND created_at_utc >= ?
      AND completed_at_utc <= ?
    ORDER BY created_at_utc
    """,
    (BATCH_UUID, aggregate['fire_timestamp_utc_start'], aggregate['fire_timestamp_utc_end']),
).fetchall()
conn.close()
ledger_cols = [
    'id', 'batch_id', 'api_call_kind', 'backend_kind', 'call_role', 'status',
    'estimated_cost', 'actual_cost', 'input_tokens', 'output_tokens',
    'created_at_utc', 'completed_at_utc', 'notes',
]
ledger_df = pd.DataFrame(ledger_rows, columns=ledger_cols)
assert len(ledger_df) == N
assert (ledger_df['backend_kind'] == 'd7b_critic').all()
assert (ledger_df['call_role'] == 'critique').all()
assert (ledger_df['api_call_kind'] == 'd7b_critic_live').all()
assert (ledger_df['status'] == 'completed').all()
assert ledger_df['estimated_cost'].map(lambda x: approx_equal(x, 0.05)).all()

per_call_by_order = {rec['firing_order']: rec for rec in per_calls}
ledger_actual_sum = round(float(ledger_df['actual_cost'].sum()), 6)
ledger_estimated_sum = round(float(ledger_df['estimated_cost'].sum()), 6)
assert ledger_actual_sum == aggregate['total_actual_cost_usd']
assert ledger_estimated_sum == aggregate['total_estimated_cost_usd']

for _, row in ledger_df.iterrows():
    m_order = re.search(r'firing_order=(\d+)', row['notes'])
    m_pos = re.search(r'position=(\d+)', row['notes'])
    assert m_order and m_pos, f'Missing firing_order/position in ledger notes: {row["notes"]}'
    order = int(m_order.group(1))
    rec = per_call_by_order[order]
    assert int(m_pos.group(1)) == rec['candidate_position']
    assert rec['ledger_row']['row_id'] == row['id']
    assert approx_equal(rec['actual_cost_usd'], row['actual_cost'], tol=1e-9)
    assert rec['critic_result']['d7b_input_tokens'] == row['input_tokens']
    assert rec['critic_result']['d7b_output_tokens'] == row['output_tokens']

display(ledger_df)


,id,batch_id,api_call_kind,backend_kind,call_role,status,estimated_cost,actual_cost,input_tokens,output_tokens,created_at_utc,completed_at_utc,notes
0,fb3eeb31-6e1a-4f07-b4be-73b14cfb1556,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.011037,1909,354,2026-04-19T14:25:43.029Z,2026-04-19T14:25:52.680Z,"Stage 2c live, position=17, firing_order=1"
1,dcc89d70-535f-45f1-b564-2ea2f94b935f,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.011943,2041,388,2026-04-19T14:25:57.688Z,2026-04-19T14:26:08.185Z,"Stage 2c live, position=22, firing_order=2"
2,37d90694-735c-44c5-ba23-b1c785c50b5f,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.011280,2180,316,2026-04-19T14:26:13.197Z,2026-04-19T14:26:22.824Z,"Stage 2c live, position=27, firing_order=3"
3,075f4bfa-0a1e-46fd-b072-ff03896311c9,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.011859,2278,335,2026-04-19T14:26:27.834Z,2026-04-19T14:26:36.443Z,"Stage 2c live, position=32, firing_order=4"
4,b2936740-443e-4c30-9db8-7550d8c4782a,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014727,2969,388,2026-04-19T14:26:41.452Z,2026-04-19T14:26:50.487Z,"Stage 2c live, position=62, firing_order=5"
5,96efc334-e73f-4ed4-9e85-2f96d85b31ca,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014223,3206,307,2026-04-19T14:26:55.496Z,2026-04-19T14:27:04.689Z,"Stage 2c live, position=72, firing_order=6"
6,ce56b686-ff9d-4b80-92fd-a4dd4907d5ea,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014331,3192,317,2026-04-19T14:27:09.699Z,2026-04-19T14:27:19.969Z,"Stage 2c live, position=73, firing_order=7"
7,22c69d43-fd71-468f-a169-bbb59063018a,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014937,3204,355,2026-04-19T14:27:24.977Z,2026-04-19T14:27:34.404Z,"Stage 2c live, position=74, firing_order=8"
8,06da54b6-d7c0-4012-a5b6-d72b52902064,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014784,3273,331,2026-04-19T14:27:39.416Z,2026-04-19T14:27:48.639Z,"Stage 2c live, position=77, firing_order=9"
9,d1d6803b-e2e1-42e6-9e72-accc24db705f,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014922,3444,306,2026-04-19T14:27:53.652Z,2026-04-19T14:28:02.668Z,"Stage 2c live, position=83, firing_order=10"


## BY. Abort Rule Recalculation Audit

Recalculate the Stage 2c abort taxonomy from persisted per-call records. Rule (c) uses scope-lock primacy: content-level errors `>= 4` only at `K >= 8`.


In [8]:
abort_triggers = []
cumulative_cost = 0.0
for idx, rec in enumerate(per_calls, 1):
    current_cost = float(rec['actual_cost_usd'] or 0)
    cumulative_cost += current_cost
    if current_cost > 0.05:
        abort_triggers.append((idx, 'rule_d_per_call_cost'))
    if cumulative_cost > 0.50:
        abort_triggers.append((idx, 'rule_e_cumulative_cost'))
    if idx >= 2:
        last_two = per_calls[idx - 2:idx]
        if all(
            r['critic_status'] == 'd7b_error'
            and r.get('d7b_error_category') == 'api_level'
            and float(r.get('actual_cost_usd') or 0) == 0
            for r in last_two
        ):
            abort_triggers.append((idx, 'rule_a_consecutive_api'))
    if idx >= 3:
        error_count = sum(1 for r in per_calls[:idx] if r['critic_status'] == 'd7b_error')
        if error_count / idx > 0.40:
            abort_triggers.append((idx, 'rule_b_cumulative_error_rate'))
    if idx >= 8:
        content_errors = sum(
            1 for r in per_calls[:idx]
            if r['critic_status'] == 'd7b_error' and r.get('d7b_error_category') == 'content_level'
        )
        if content_errors >= 4:
            abort_triggers.append((idx, 'rule_c_content_level_threshold'))

abort_audit = {
    'rule_a_consecutive_api': not any(r == 'rule_a_consecutive_api' for _, r in abort_triggers),
    'rule_b_cumulative_error_rate': not any(r == 'rule_b_cumulative_error_rate' for _, r in abort_triggers),
    'rule_c_content_level_threshold': not any(r == 'rule_c_content_level_threshold' for _, r in abort_triggers),
    'rule_d_per_call_cost': not any(r == 'rule_d_per_call_cost' for _, r in abort_triggers),
    'rule_e_cumulative_cost': not any(r == 'rule_e_cumulative_cost' for _, r in abort_triggers),
}
assert all(abort_audit.values()), abort_triggers
assert aggregate['sequence_aborted'] is False
abort_rule_df = pd.DataFrame([{'rule': k, 'not_triggered': v} for k, v in abort_audit.items()])
display(abort_rule_df)


,rule,not_triggered
0,rule_a_consecutive_api,True
1,rule_b_cumulative_error_rate,True
2,rule_c_content_level_threshold,True
3,rule_d_per_call_cost,True
4,rule_e_cumulative_cost,True


## BZ. Stage 2d Green-Light Criteria Audit

Adjudicate the four green-light gates: cost, reasoning length, pattern-or-named-ambiguity, and recurring error class.


In [9]:
def reconcile_label(label: str, svr: float | None) -> str:
    if svr is None or label == 'neutral':
        return 'n/a'
    if label == 'agreement_expected':
        return 'supports' if svr >= 0.5 else 'contradicts'
    if label == 'divergence_expected':
        return 'supports' if svr < 0.5 else 'contradicts'
    return 'n/a'

recon_rows = []
for rec in per_calls:
    svr = d7b_scores(rec).get('structural_variant_risk')
    recon_rows.append({
        'position': rec['candidate_position'],
        'label': rec['pre_registered_label'],
        'svr': svr,
        'reconcile': reconcile_label(rec['pre_registered_label'], svr),
        'overlap': rec['candidate_position'] in OVERLAP_POSITIONS,
    })
recon_df = pd.DataFrame(recon_rows)

agreement_table = recon_df[recon_df['label'] == 'agreement_expected'].copy()
divergence_table = recon_df[recon_df['label'] == 'divergence_expected'].copy()
neutral_table = recon_df[recon_df['label'] == 'neutral'].copy()
print('agreement_expected group')
display(agreement_table)
print('divergence_expected group')
display(divergence_table)
print('neutral group')
display(neutral_table)

directional = recon_df[recon_df['label'].isin(['agreement_expected', 'divergence_expected'])]
directional_support_count = int((directional['reconcile'] == 'supports').sum())
pattern_found = directional_support_count >= 7
if pattern_found:
    ambiguity_source = None
elif not divergence_table.empty and divergence_table['overlap'].all():
    ambiguity_source = 'overlap confounding'
elif selection_label_counts.get('divergence_expected', 0) <= 3:
    ambiguity_source = 'label sparsity'
elif max(selection_theme_counts.values()) / N > 0.5:
    ambiguity_source = 'theme imbalance'
else:
    ambiguity_source = 'test-retest variance'
pattern_gate_pass = pattern_found or ambiguity_source in ALLOWED_AMBIGUITY_SOURCES

cost_gate_pass = aggregate['total_actual_cost_usd'] <= aggregate['total_estimated_cost_usd']
actual_estimated_ratio = aggregate['total_actual_cost_usd'] / aggregate['total_estimated_cost_usd']
reasoning_gate_pass = all(100 <= n <= 1500 for n in recomputed_reasoning_lengths)
reasoning_cap_hit_count = sum(1 for n in recomputed_reasoning_lengths if n > 1500)
error_count = sum(1 for s in recomputed_statuses if s == 'd7b_error')
error_gate_pass = error_count <= 2

green_light_df = pd.DataFrame([
    {'gate': 'cost', 'pass': cost_gate_pass, 'evidence': f"actual/estimated={actual_estimated_ratio:.3f}"},
    {'gate': 'reasoning length', 'pass': reasoning_gate_pass and reasoning_cap_hit_count <= 1, 'evidence': f'cap_hits={reasoning_cap_hit_count}; range={min(recomputed_reasoning_lengths)}..{max(recomputed_reasoning_lengths)}'},
    {'gate': 'pattern or named ambiguity', 'pass': pattern_gate_pass, 'evidence': 'pattern found' if pattern_found else f'ambiguity source: {ambiguity_source}'},
    {'gate': 'recurring error class', 'pass': error_gate_pass, 'evidence': f'd7b_error_count={error_count}'},
])
assert green_light_df['pass'].all(), green_light_df.loc[~green_light_df['pass']]
display(green_light_df)


agreement_expected group


,position,label,svr,reconcile,overlap
2,27,agreement_expected,0.85,supports,False
10,97,agreement_expected,0.95,supports,True
11,102,agreement_expected,0.95,supports,False
12,107,agreement_expected,0.95,supports,False
13,112,agreement_expected,0.85,supports,False
17,147,agreement_expected,0.95,supports,False
18,152,agreement_expected,0.95,supports,False
19,162,agreement_expected,0.90,supports,False


divergence_expected group


,position,label,svr,reconcile,overlap
0,17,divergence_expected,0.85,contradicts,True
6,73,divergence_expected,0.95,contradicts,True
7,74,divergence_expected,0.75,contradicts,True


neutral group


,position,label,svr,reconcile,overlap
1,22,neutral,0.85,n/a,False
3,32,neutral,0.90,n/a,False
4,62,neutral,0.95,n/a,False
5,72,neutral,0.75,n/a,False
8,77,neutral,0.95,n/a,False
9,83,neutral,0.75,n/a,False
14,117,neutral,0.85,n/a,False
15,138,neutral,0.30,n/a,True
16,143,neutral,0.15,n/a,False


,gate,pass,evidence
0,cost,True,actual/estimated=0.316
1,reasoning length,True,cap_hits=0; range=960..1245
2,pattern or named ambiguity,True,pattern found
3,recurring error class,True,d7b_error_count=0


## CA. Test-Retest Reconciliation for Stage 2b Overlap

Compare the five overlap positions against Stage 2b on each D7b score axis. This is bookkeeping for interpretation, not an implementation gate.


In [10]:
assert STAGE2B_AGGREGATE_PATH.exists(), f'Missing Stage 2b aggregate: {STAGE2B_AGGREGATE_PATH}'
stage2b_aggregate = load_json(STAGE2B_AGGREGATE_PATH)
stage2b_by_pos = {rec['candidate_position']: rec for rec in stage2b_aggregate['per_call_records']}
stage2c_by_pos = {rec['candidate_position']: rec for rec in per_calls}
axes = ['semantic_plausibility', 'semantic_theme_alignment', 'structural_variant_risk']

tr_rows = []
axis_agree = []
full_vector_count = 0
for pos in sorted(OVERLAP_POSITIONS):
    b = stage2b_by_pos.get(pos)
    c = stage2c_by_pos.get(pos)
    assert b is not None, f'Missing Stage 2b overlap record for {pos}'
    assert c is not None, f'Missing Stage 2c overlap record for {pos}'
    b_scores = d7b_scores(b)
    c_scores = d7b_scores(c)
    diffs = {axis: abs(float(c_scores[axis]) - float(b_scores[axis])) for axis in axes}
    agrees = {axis: diffs[axis] <= 0.1 for axis in axes}
    axis_agree.extend(agrees.values())
    if all(agrees.values()):
        full_vector_count += 1
    tr_rows.append({
        'position': pos,
        'label': c['pre_registered_label'],
        'stage2b_plaus': b_scores['semantic_plausibility'],
        'stage2c_plaus': c_scores['semantic_plausibility'],
        'plaus_diff': round(diffs['semantic_plausibility'], 6),
        'stage2b_align': b_scores['semantic_theme_alignment'],
        'stage2c_align': c_scores['semantic_theme_alignment'],
        'align_diff': round(diffs['semantic_theme_alignment'], 6),
        'stage2b_svr': b_scores['structural_variant_risk'],
        'stage2c_svr': c_scores['structural_variant_risk'],
        'svr_diff': round(diffs['structural_variant_risk'], 6),
        'full_vector_within_0_1': all(agrees.values()),
    })

test_retest_df = pd.DataFrame(tr_rows)
per_axis_agreement_rate = sum(axis_agree) / len(axis_agree)
test_retest_recorded = len(test_retest_df) == 5 and len(axis_agree) == 15
assert test_retest_recorded
display(test_retest_df)
print('per-axis agreement rate within +/-0.1:', per_axis_agreement_rate)
print('full-vector agreement count:', full_vector_count, 'of 5')


,position,label,stage2b_plaus,stage2c_plaus,plaus_diff,stage2b_align,stage2c_align,align_diff,stage2b_svr,stage2c_svr,svr_diff,full_vector_within_0_1
0,17,divergence_expected,0.75,0.75,0.0,0.85,0.85,0.00,0.85,0.85,0.00,True
1,73,divergence_expected,0.75,0.75,0.0,0.85,0.85,0.00,0.85,0.95,0.10,True
2,74,divergence_expected,0.75,0.75,0.0,0.85,0.90,0.05,0.65,0.75,0.10,True
3,97,agreement_expected,0.75,0.75,0.0,0.90,0.90,0.00,0.95,0.95,0.00,True
4,138,neutral,0.75,0.75,0.0,0.90,0.90,0.00,0.15,0.30,0.15,False


per-axis agreement rate within +/-0.1: 0.9333333333333333
full-vector agreement count: 4 of 5


## CB. Neutral-Group Aggregate Prediction Reconciliation

Reconcile the pre-registered neutral-group prediction against the nine observed neutral `structural_variant_risk` values.


In [11]:
assert neutral_group_body is not None and len(neutral_non_ws) >= 50
display(Markdown('### Pre-registered neutral-group prediction\n\n> ' + neutral_group_body.strip().replace('\n', '\n> ')))

neutral_records = [rec for rec in per_calls if rec['pre_registered_label'] == 'neutral']
neutral_svrs = [float(d7b_scores(rec)['structural_variant_risk']) for rec in neutral_records]
neutral_low_count = sum(1 for v in neutral_svrs if v < 0.5)
neutral_high_count = sum(1 for v in neutral_svrs if v >= 0.5)
neutral_median = median(neutral_svrs)
neutral_stats = {
    'count': len(neutral_svrs),
    'min': min(neutral_svrs),
    'max': max(neutral_svrs),
    'median': neutral_median,
    'mean': mean(neutral_svrs),
    'below_0_5_count': neutral_low_count,
    'at_or_above_0_5_count': neutral_high_count,
}
median_supported = 0.35 <= neutral_median <= 0.65
straddle_supported = neutral_low_count >= 2 and neutral_high_count >= 2
if median_supported and straddle_supported:
    neutral_prediction_result = 'support'
elif median_supported or straddle_supported:
    neutral_prediction_result = 'partial'
else:
    neutral_prediction_result = 'contradict'
neutral_reconciled = neutral_prediction_result in {'support', 'partial', 'contradict'}
assert len(neutral_svrs) == selection_label_counts['neutral'] == 9
assert neutral_reconciled
neutral_df = pd.DataFrame([
    {'position': rec['candidate_position'], 'svr': float(d7b_scores(rec)['structural_variant_risk'])}
    for rec in neutral_records
])
display(neutral_df)
display(pd.DataFrame([neutral_stats | {'prediction_result': neutral_prediction_result}]))


### Pre-registered neutral-group prediction

> Stage 2c includes nine `neutral`-labeled candidates (positions 22, 32, 62,
> 72, 77, 83, 117, 138, 143). Per scope lock §Lock 8, these candidates do
> not carry per-candidate polarity predictions; the replication mechanism
> for the neutral group is an **aggregate-level** pre-registered claim.
> 
> **Pre-fire claim (falsifiable):** Across the 9 neutral candidates, the
> empirical `structural_variant_risk` distribution is predicted to have a
> moderate-to-high center rather than collapsing toward either extreme. The
> median `structural_variant_risk` is predicted to fall inside the interval
> `[0.45, 0.70]`. At least 1 of the 9 scores is predicted to fall below
> 0.5, and at least 4 of the 9 scores are predicted to fall at or above
> 0.5. If the observed neutral-group median falls outside `[0.45, 0.70]`,
> or if no scores fall below 0.5, or if fewer than 4 scores fall at or above
> 0.5, the pre-registered aggregate claim is considered **falsified**.
> 
> This aggregate prediction reflects the full set of per-candidate informal
> neutral leans completed before fire: most neutral candidates lean
> moderate-to-high, but at least one below-threshold case is still expected.
> The goal is to preserve a falsifiable distribution-shape claim without
> imposing a stronger bimodal or monotonic subgroup hypothesis than the
> candidate-level judgments support.

,position,svr
0,22,0.85
1,32,0.90
2,62,0.95
3,72,0.75
4,77,0.95
5,83,0.75
6,117,0.85
7,138,0.30
8,143,0.15


,count,min,max,median,mean,below_0_5_count,at_or_above_0_5_count,prediction_result
0,9,0.15,0.95,0.85,0.716667,2,7,partial


## CC. D7 Stage 2c Verdict

Collapse implementation integrity, green-light criteria, test-retest bookkeeping, and neutral-group reconciliation into the final Stage 2c decision.


In [12]:
artifact_integrity_pass = all([
    len(candidates) == N,
    aggregate['stage_label'] == 'd7_stage2c',
    aggregate['record_version'] == '1.0',
    aggregate['completed_call_count'] == N,
    aggregate['sequence_aborted'] is False,
    aggregate['abort_reason'] is None,
    all(abort_audit.values()),
    len(ledger_df) == N,
    RAW_CRITIC_DIR.exists(),
    archive_dir.exists(),
])

verdict_rows = [
    {'gate': 'Artifact integrity', 'result': 'PASS' if artifact_integrity_pass else 'FAIL', 'evidence': 'sections BS-BY'},
    {'gate': 'Cost gate', 'result': 'PASS' if cost_gate_pass else 'FAIL', 'evidence': f"{aggregate['total_actual_cost_usd']} <= {aggregate['total_estimated_cost_usd']}"},
    {'gate': 'Reasoning-length gate', 'result': 'PASS' if reasoning_gate_pass else 'FAIL', 'evidence': f'{min(recomputed_reasoning_lengths)}..{max(recomputed_reasoning_lengths)}'},
    {'gate': 'Error-class gate', 'result': 'PASS' if error_gate_pass else 'FAIL', 'evidence': f'd7b_error_count={error_count}'},
    {'gate': 'Pattern/ambiguity gate', 'result': 'PASS' if pattern_gate_pass else 'FAIL', 'evidence': 'pattern found' if pattern_found else f'ambiguity source: {ambiguity_source}'},
    {'gate': 'Test-retest recorded', 'result': 'PASS' if test_retest_recorded else 'FAIL', 'evidence': f'per-axis={per_axis_agreement_rate:.3f}; full-vector={full_vector_count}/5'},
    {'gate': 'Neutral-group prediction reconciled', 'result': 'PASS' if neutral_reconciled else 'FAIL', 'evidence': neutral_prediction_result},
]
verdict_df = pd.DataFrame(verdict_rows)
stage2c_overall_pass = (verdict_df['result'] == 'PASS').all()
verdict_df.loc[len(verdict_df)] = {
    'gate': 'Stage 2c overall',
    'result': 'PASS' if stage2c_overall_pass else 'FAIL',
    'evidence': 'green-light Stage 2d' if stage2c_overall_pass else 'block Stage 2d until failed gates are patched',
}
display(verdict_df)
assert stage2c_overall_pass, verdict_df.loc[verdict_df['result'] != 'PASS']
print('PASS and green-light Stage 2d')


,gate,result,evidence
0,Artifact integrity,PASS,sections BS-BY
1,Cost gate,PASS,0.315765 <= 1.0
2,Reasoning-length gate,PASS,960..1245
3,Error-class gate,PASS,d7b_error_count=0
4,Pattern/ambiguity gate,PASS,pattern found
5,Test-retest recorded,PASS,per-axis=0.933; full-vector=4/5
6,Neutral-group prediction reconciled,PASS,partial
7,Stage 2c overall,PASS,green-light Stage 2d


PASS and green-light Stage 2d
